# HC3 Logistic Regression Baseline

This notebook trains an answer-level human-vs.-ChatGPT detector with word and character TF-IDF plus simple writing-style features. It uses the existing question-level train/validation/test split so answers to the same question cannot cross split boundaries.

Run `data_cleaning.ipynb` and `dataset_splitting.ipynb` first. The vectorizers are fit only on training answers; validation and test answers are used only for evaluation. The test set is evaluated once at the end.


In [ ]:
from ast import literal_eval
import re

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             f1_score, accuracy_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## Load the question-level splits and expand answers

Each original question can have multiple human and ChatGPT answers. Expand them after splitting, assigning one label per answer: `0` for human and `1` for ChatGPT. Keep question IDs and domain labels for grouping and error analysis.


In [ ]:
SPLIT_DIR = "../data/processed"


def parse_answers(value):
    # CSV stores answer lists as Python-style string representations.
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    parsed = literal_eval(value)
    if not isinstance(parsed, list):
        raise ValueError("Expected an answer list in the HC3 CSV")
    return parsed


def expand_answers(frame):
    records = []
    for row in frame[["id", "source", "human_answers", "chatgpt_answers"]].itertuples(index=False):
        question_id, source, human_answers, chatgpt_answers = row
        for label, answers in ((0, human_answers), (1, chatgpt_answers)):
            for answer in parse_answers(answers):
                if isinstance(answer, str) and answer.strip():
                    records.append({"id": question_id, "source": source,
                                    "text": answer, "label": label})
    return pd.DataFrame.from_records(records)

train_questions = pd.read_csv(f"{SPLIT_DIR}/train.csv")
validation_questions = pd.read_csv(f"{SPLIT_DIR}/validation.csv")
test_questions = pd.read_csv(f"{SPLIT_DIR}/test.csv")

train = expand_answers(train_questions)
validation = expand_answers(validation_questions)
test = expand_answers(test_questions)

print(f"Answer rows — train: {len(train):,}; validation: {len(validation):,}; test: {len(test):,}")
print("Train label counts:\n", train["label"].value_counts().sort_index())


## Build interpretable style features

These features capture basic length and punctuation patterns. They complement lexical TF-IDF and are computed per answer without looking at labels or other rows.


In [ ]:
WORD_PATTERN = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)*", re.UNICODE)


def add_style_features(frame):
    frame = frame.copy()
    text = frame["text"]
    frame["char_count"] = text.str.len()
    frame["word_count"] = text.map(lambda value: len(WORD_PATTERN.findall(value)))
    frame["sentence_count"] = text.map(
        lambda value: sum(bool(part.strip()) for part in re.split(r"(?<=[.!?])\s+", value.strip()))
    )
    frame["paragraph_count"] = text.map(
        lambda value: sum(bool(part.strip()) for part in re.split(r"\n[ \t]*\n+", value.strip()))
    )
    frame["comma_count"] = text.str.count(",")
    frame["question_mark_count"] = text.str.count(r"\?")
    frame["exclamation_count"] = text.str.count("!")
    frame["uppercase_ratio"] = text.map(
        lambda value: sum(char.isupper() for char in value) / max(len(value), 1)
    )
    frame["punctuation_ratio"] = text.map(
        lambda value: sum(not char.isalnum() and not char.isspace() for char in value)
        / max(len(value), 1)
    )
    return frame

train = add_style_features(train)
validation = add_style_features(validation)
test = add_style_features(test)

STYLE_FEATURES = ["char_count", "word_count", "sentence_count", "paragraph_count",
                  "comma_count", "question_mark_count", "exclamation_count",
                  "uppercase_ratio", "punctuation_ratio"]


## Fit the baseline

Word n-grams capture phrase patterns; character n-grams capture spelling and formatting patterns. `class_weight="balanced"` adjusts for the unequal number of answers in each class. The validation split is kept available for model choices; this baseline is evaluated on it for an initial check, then on the held-out test set.


In [ ]:
features = ColumnTransformer(
    transformers=[
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                                  max_features=100_000, sublinear_tf=True), "text"),
        ("char", TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2,
                                  max_features=100_000, sublinear_tf=True), "text"),
        ("style", StandardScaler(), STYLE_FEATURES),
    ],
    remainder="drop",
)
model = Pipeline([
    ("features", features),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced",
                                       random_state=42, n_jobs=-1)),
])

model.fit(train, train["label"])
validation_predictions = model.predict(validation)
print("Validation accuracy:", round(accuracy_score(validation["label"], validation_predictions), 4))
print("Validation macro-F1:", round(f1_score(validation["label"], validation_predictions,
                                                average="macro"), 4))


## Final held-out test evaluation

Report both accuracy and macro-F1, plus per-class precision/recall and a confusion matrix. The majority-class baseline provides a simple reference point for accuracy.


In [ ]:
test_predictions = model.predict(test)
majority_label = train["label"].mode().iloc[0]
majority_predictions = [majority_label] * len(test)

print("Majority-class accuracy:", round(accuracy_score(test["label"], majority_predictions), 4))
print("Model accuracy:", round(accuracy_score(test["label"], test_predictions), 4))
print("Model macro-F1:", round(f1_score(test["label"], test_predictions, average="macro"), 4))
print("\nPer-class metrics:\n", classification_report(
    test["label"], test_predictions, labels=[0, 1], target_names=["human", "ChatGPT"],
    zero_division=0,
))

ConfusionMatrixDisplay.from_predictions(
    test["label"], test_predictions, labels=[0, 1], display_labels=["human", "ChatGPT"],
    cmap="Blues", values_format="d",
)
plt.title("HC3 test set: human vs. ChatGPT")
plt.tight_layout()
plt.show()


## Check performance by domain

Domain breakdowns can reveal that overall performance hides differences between finance, medicine, open Q&A, ELI5, and computer-science answers. Each test question remains grouped in one split.


In [ ]:
domain_results = []
for source, rows in test.assign(prediction=test_predictions).groupby("source"):
    domain_results.append({
        "source": source,
        "answer_count": len(rows),
        "accuracy": accuracy_score(rows["label"], rows["prediction"]),
        "macro_f1": f1_score(rows["label"], rows["prediction"], average="macro"),
    })

domain_results = pd.DataFrame(domain_results).sort_values("source")
display(domain_results.round(4))
